# Prediction Interval Estimation & Uncertainty Quantification
## Applied to grocery store income prediction under domain shift

When predicting income for an unknown target store, we do not have target labels
to evaluate our predictions. This notebook asks: **can we estimate our prediction
error in advance, without seeing the true values?**

Four methods are implemented and compared:

| # | Method | Type | Key idea |
|---|---|---|---|
| M1 | Source CV residuals | Point estimate | CV error on labeled source data |
| M2 | Shift-adjusted interval | Point estimate | Inflate M1 by KS-based factor |
| M3 | Conformal prediction | Coverage interval | Distribution-free ±interval guarantee |
| M4 | Density-ratio reweighting | Point estimate | Reweight source errors by target similarity |

**Source:** Kroger + Safeway + Walmart (median tier, $30k-$105k)  
**Targets:** All 5 stores in Leave-One-Store-Out CV

## 1. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.stats import rankdata, spearmanr, ks_2samp
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

STORES  = ["whole_foods","kroger","safeway","walmart","thrift_store"]
LABELS  = ["Whole Foods","Kroger","Safeway","Walmart","Thrift"]
TIERS   = ["High","Median","Median","Median","Low"]
COLORS  = ["#185FA5","#1D9E75","#0F6E56","#854F0B","#A32D2D"]

FEAT_COLS = ["age","visits_per_month","avg_basket_usd","monthly_spend_usd",
             "grocery_pct","electronics_pct","apparel_pct","home_pct",
             "private_label_pct","online_orders_pct","coupon_usage_pct",
             "loyalty_score","segment_enc"]
PROXY_IDX = [2, 9]
BEST_ALPHAS = [0.1, 0.3, 0.5, 0.7]

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5, "font.size": 10,
})
print("Setup complete.")

## 2. Load data and build shared infrastructure

In [ ]:
df = pd.read_csv("grocery_all_stores.csv")
le = LabelEncoder(); le.fit(df["segment"]); df["segment_enc"] = le.transform(df["segment"])

TRUE_RANGES = {"whole_foods":(85000,160000),"kroger":(55000,105000),
               "safeway":(45000,95000),"walmart":(30000,80000),"thrift_store":(10000,45000)}

sc_all   = StandardScaler()
X_all_sc = sc_all.fit_transform(df[FEAT_COLS].values.astype(float))
pca_ref  = PCA(n_components=6, random_state=42)
X_all_pca = pca_ref.fit_transform(X_all_sc)

ref_lib = {}
for s in STORES:
    mask = df["store"] == s
    ref_lib[s] = {"centroid": X_all_pca[mask].mean(axis=0),
                  "income_mean": np.mean(TRUE_RANGES[s])}

def make_bridge(Xs, Xt, yr, alphas=BEST_ALPHAS, n=400, seed=42):
    rng = np.random.RandomState(seed); bX, by = [], []
    for a in alphas:
        for _ in range(n // len(alphas)):
            i, j = rng.randint(len(Xs)), rng.randint(len(Xt))
            bX.append((1-a)*Xs[i] + a*Xt[j])
            by.append((1-a)*yr[i]  + a*0.5)
    return np.array(bX), np.array(by)

def get_range(Xs, Xt, ys, ts, refs):
    preg = LinearRegression().fit(Xs[:,PROXY_IDX], ys)
    gap  = preg.predict(Xt[:,PROXY_IDX]).mean() - preg.predict(Xs[:,PROXY_IDX]).mean()
    cen  = X_all_pca[df["store"]==ts].mean(axis=0).reshape(1,-1)
    dists = {s: cdist(cen, ref_lib[s]["centroid"].reshape(1,-1), "euclidean")[0,0]
             for s in refs}
    nn   = min(dists, key=dists.get)
    bl   = 0.4*ref_lib[nn]["income_mean"] + 0.6*(preg.predict(Xs[:,PROXY_IDX]).mean()+gap)
    span = (ys.max()-ys.min())/2 * 1.1
    return max(0, round((bl-span)/1000)*1000), round((bl+span)/1000)*1000

def run_model(Xs, Xt, ys, lo, hi):
    yr = (rankdata(ys)-1)/(len(ys)-1)
    bX, by = make_bridge(Xs, Xt, yr)
    m = Ridge(alpha=10.0)
    m.fit(np.vstack([Xs,bX]), np.concatenate([yr,by]))
    return lo + np.clip(m.predict(Xt),0,1)*(hi-lo), m

print("Infrastructure ready. Stores:", STORES)

## 3. What is uncertainty quantification and why does it matter?

A prediction is only half the story. Knowing that a model predicts income of $75,000
for a customer is useful. Knowing that this prediction is reliable within ±$5,000
versus ±$25,000 is what determines whether the prediction is *actionable*.

In domain adaptation, the challenge is compounded: the model was trained on data
from a different domain, so even if it performs well on average, individual
predictions may be far off.

**Three questions we want to answer without using target labels:**
1. What is our expected average error (MAE estimate)?
2. What ±interval will contain the true value X% of the time?
3. How does our confidence vary across customers within the target store?

In [ ]:
# Run the full pipeline for all 5 stores and collect everything we need
results = {}

for ts in STORES:
    src = df[df["store"]!=ts].copy()
    tgt = df[df["store"]==ts].copy()
    Xsr = src[FEAT_COLS].values.astype(float)
    Xtr = tgt[FEAT_COLS].values.astype(float)
    ys  = src["income_usd"].values
    yt  = tgt["income_usd"].values

    sc = StandardScaler(); Xs = sc.fit_transform(Xsr); Xt = sc.transform(Xtr)
    refs = [s for s in STORES if s != ts]
    lo, hi = get_range(Xs, Xt, ys, ts, refs)

    preds, model = run_model(Xs, Xt, ys, lo, hi)
    actual_errors = np.abs(yt - preds)
    actual_mae    = actual_errors.mean()

    # KS shift
    avg_ks = np.mean([ks_2samp(Xs[:,i], Xt[:,i]).statistic
                      for i in range(Xs.shape[1])])

    results[ts] = {
        "Xs": Xs, "Xt": Xt, "ys": ys, "yt": yt,
        "preds": preds, "actual_errors": actual_errors,
        "actual_mae": actual_mae, "lo": lo, "hi": hi,
        "model": model, "avg_ks": avg_ks,
        "sc": sc,
    }
    print(f"  {LABELS[STORES.index(ts)]:<15}  actual MAE=${actual_mae:>8,.0f}  "
          f"range=${lo:,}-${hi:,}  avg_KS={avg_ks:.3f}")

## 4. Method 1 — Source cross-validation residuals

**Core idea:** Run K-fold CV on the labeled source data.
Collect |predicted - actual| on held-out folds.
Use the mean of these as the predicted MAE for any target.

**Formula:**
```
for each fold k:
    train rank model on source[k]
    residuals_k = |y_val - predict(X_val, lo, hi)|
predicted_MAE = mean(all_residuals)
```

**When it works:** When source and target have similar distributions (low KS).
The source CV error is a fair proxy for target error.

**When it fails:** For high-shift targets (WF, Thrift), the source CV error
reflects within-source-range difficulty — not the extrapolation difficulty
of predicting outside the training income range.

In [ ]:
def method1_source_cv(Xs, ys, lo, hi, k=5):
    kf   = KFold(n_splits=k, shuffle=True, random_state=42)
    resid = []
    for tr, val in kf.split(Xs):
        yr_tr = (rankdata(ys[tr])-1)/(len(ys[tr])-1)
        m = Ridge(alpha=10.0); m.fit(Xs[tr], yr_tr)
        pv = lo + np.clip(m.predict(Xs[val]),0,1)*(hi-lo)
        resid.extend(np.abs(ys[val]-pv))
    return np.array(resid)

# Compute for all stores
m1_results = {}
for ts in STORES:
    r = results[ts]
    resid = method1_source_cv(r["Xs"], r["ys"], r["lo"], r["hi"])
    m1_results[ts] = {"est_mae": resid.mean(), "est_std": resid.std(),
                      "residuals": resid}

# ── Plot: estimated vs actual MAE ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Method 1 - Source CV residuals: estimated vs actual MAE", fontweight="bold")

x = np.arange(len(STORES))
est  = [m1_results[s]["est_mae"] for s in STORES]
act  = [results[s]["actual_mae"]  for s in STORES]
errs = [m1_results[s]["est_std"]  for s in STORES]

ax = axes[0]
bars1 = ax.bar(x - 0.2, [a/1000 for a in act], 0.35, label="Actual MAE",
               color=[c+"99" for c in COLORS], edgecolor=COLORS, linewidth=1.5)
bars2 = ax.bar(x + 0.2, [e/1000 for e in est], 0.35, label="M1 Estimated",
               color=COLORS, edgecolor="white", alpha=0.7,
               yerr=[e/1000 for e in errs], capsize=4, error_kw={"elinewidth":1.5})
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("MAE ($k)"); ax.set_title("Estimated vs Actual MAE", fontweight="bold")
ax.legend()
for i, (a, e) in enumerate(zip(act, est)):
    ax.text(i, max(a, e)/1000 + 1.5, f"err=${abs(a-e)/1000:.1f}k",
            ha="center", fontsize=8, color="black")

# ── Plot: residual distributions per store ────────────────────────────────────
ax = axes[1]
parts = ax.violinplot([m1_results[s]["residuals"]/1000 for s in STORES],
                       positions=x, widths=0.6, showmedians=True)
for i, (pc, col) in enumerate(zip(parts["bodies"], COLORS)):
    pc.set_facecolor(col); pc.set_alpha(0.6)
for part in ["cbars","cmins","cmaxes","cmedians"]:
    parts[part].set_color("gray"); parts[part].set_linewidth(1)
for i, ts in enumerate(STORES):
    ax.scatter(i, results[ts]["actual_mae"]/1000, color=COLORS[i],
               s=80, marker="D", zorder=5, label="Actual MAE" if i==0 else "")
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Absolute error ($k)")
ax.set_title("Source CV residual distributions (diamond = actual target MAE)", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Summary:")
for ts, lbl in zip(STORES, LABELS):
    act = results[ts]["actual_mae"]; est = m1_results[ts]["est_mae"]
    print(f"  {lbl:<15}  actual=${act:>8,.0f}  M1_est=${est:>8,.0f}  "
          f"error=${abs(act-est):>7,.0f}  ({abs(act-est)/act*100:.0f}% off)")

## 5. Method 2 — Shift-adjusted interval

**Core idea:** Source CV underestimates error when there is domain shift.
Inflate the M1 estimate by a factor proportional to the feature shift magnitude.

**Formula:**
```
shift_factor = 1 + avg_KS * k     (k = 2.0 is a heuristic constant)
adjusted_MAE = M1_estimated_MAE * shift_factor
```

**Interpretation:**
- KS = 0 (no shift): factor = 1.0 (no inflation)
- KS = 0.5 (large shift): factor = 2.0 (double the error estimate)
- KS = 1.0 (complete separation): factor = 3.0

**Caveat:** The k=2.0 constant is a heuristic, not a tuned value.
This method tends to over-inflate at high KS values.
Use it as an upper bound, not a precise estimate.

In [ ]:
def method2_shift_adjusted(m1_mae, avg_ks, k=2.0):
    factor = 1 + avg_ks * k
    return m1_mae * factor, factor

m2_results = {}
for ts in STORES:
    avg_ks = results[ts]["avg_ks"]
    m2_mae, factor = method2_shift_adjusted(m1_results[ts]["est_mae"], avg_ks)
    m2_results[ts] = {"est_mae": m2_mae, "factor": factor, "avg_ks": avg_ks}

# ── Plot: how the shift factor amplifies the estimate ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle("Method 2 - Shift-adjusted interval: how KS inflates the estimate",
             fontweight="bold")

# Left: KS vs shift factor
ax = axes[0]
ks_range = np.linspace(0, 1, 100)
for k_val, ls, lbl in [(1.0,":","k=1.0 (mild)"), (2.0,"-","k=2.0 (default)"),
                        (3.0,"--","k=3.0 (aggressive)")]:
    ax.plot(ks_range, 1+ks_range*k_val, linestyle=ls, linewidth=2, label=lbl)
for ts, lbl, col in zip(STORES, LABELS, COLORS):
    ax.axvline(m2_results[ts]["avg_ks"], color=col, linewidth=1, alpha=0.7)
    ax.text(m2_results[ts]["avg_ks"]+0.01, 1.1, lbl.split()[0],
            color=col, fontsize=7.5, rotation=90)
ax.set_xlabel("Avg KS statistic"); ax.set_ylabel("Inflation factor")
ax.set_title("KS -> inflation factor", fontweight="bold"); ax.legend(fontsize=8)

# Middle: M1 vs M2 vs actual
x = np.arange(len(STORES))
ax = axes[1]
w = 0.25
ax.bar(x-w, [results[s]["actual_mae"]/1000 for s in STORES], w,
       label="Actual", color=[c+"99" for c in COLORS], edgecolor=COLORS, linewidth=1.5)
ax.bar(x,   [m1_results[s]["est_mae"]/1000 for s in STORES], w,
       label="M1 (source CV)", color=COLORS, alpha=0.55, edgecolor="white")
ax.bar(x+w, [m2_results[s]["est_mae"]/1000 for s in STORES], w,
       label="M2 (shift-adj)", color=COLORS, alpha=0.9, edgecolor="white",
       hatch="//")
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("MAE ($k)"); ax.set_title("M1 vs M2 vs Actual", fontweight="bold")
ax.legend(fontsize=8)

# Right: estimation error comparison
ax = axes[2]
m1_err = [abs(results[s]["actual_mae"]-m1_results[s]["est_mae"])/1000 for s in STORES]
m2_err = [abs(results[s]["actual_mae"]-m2_results[s]["est_mae"])/1000 for s in STORES]
ax.bar(x-0.2, m1_err, 0.35, label="M1 error", color=[c+"88" for c in COLORS],
       edgecolor=COLORS, linewidth=1.5)
ax.bar(x+0.2, m2_err, 0.35, label="M2 error", color=COLORS, edgecolor="white", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("|Estimated - Actual| MAE ($k)")
ax.set_title("|Estimation error| M1 vs M2", fontweight="bold"); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

print("M2 results:")
for ts, lbl in zip(STORES, LABELS):
    print(f"  {lbl:<15}  KS={m2_results[ts]['avg_ks']:.3f}  "
          f"factor={m2_results[ts]['factor']:.2f}x  "
          f"M2_est=${m2_results[ts]['est_mae']:>8,.0f}  "
          f"actual=${results[ts]['actual_mae']:>8,.0f}")

## 6. Method 3 — Conformal prediction intervals

This is the most principled method. Conformal prediction gives a
**distribution-free coverage guarantee** without any parametric assumptions.

**Algorithm (split conformal prediction):**
1. Split source into train (80%) and calibration (20%)
2. Train the rank model on train split + bridge samples
3. For each calibration sample i, compute the **nonconformity score:**
   `alpha_i = |y_i - predicted_i|`
4. Find the empirical quantile at level `ceil((1-alpha)(1+1/n_cal)) / n_cal`:
   `q = quantile(alpha_1...alpha_n, (1-alpha)(1+1/n))`
5. For each target prediction:
   `interval = [pred - q, pred + q]`

**Coverage guarantee (under IID):**
`P(y_test in [pred - q, pred + q]) >= 1 - alpha`

Under domain shift, this guarantee is approximate — but it is still
the most honest uncertainty statement available without target labels.

**Key parameters:**
- `coverage_target`: desired coverage level (e.g. 0.90 for 90% intervals)
- `cal_fraction`: fraction of source used for calibration (default 0.20)

In [ ]:
def method3_conformal(Xs, ys, Xt, lo, hi, coverage=0.90, cal_frac=0.20, seed=42):
    rng  = np.random.RandomState(seed)
    idx  = rng.permutation(len(Xs))
    n_c  = int(len(Xs) * cal_frac)
    tr_i, cal_i = idx[:-n_c], idx[-n_c:]

    yr_tr = (rankdata(ys[tr_i])-1)/(len(ys[tr_i])-1)
    bX, by = make_bridge(Xs[tr_i], Xt, yr_tr)
    m = Ridge(alpha=10.0)
    m.fit(np.vstack([Xs[tr_i], bX]), np.concatenate([yr_tr, by]))

    pred_cal  = lo + np.clip(m.predict(Xs[cal_i]),0,1)*(hi-lo)
    nc_scores = np.abs(ys[cal_i] - pred_cal)          # nonconformity scores

    # Quantile with finite-sample correction
    q_level = min((1-coverage) * 0 + coverage * (1 + 1/n_c), 1.0)
    q = float(np.quantile(nc_scores, q_level))

    pred_tgt = lo + np.clip(m.predict(Xt),0,1)*(hi-lo)
    intervals = np.column_stack([pred_tgt - q, pred_tgt + q])

    return {"q": q, "nc_scores": nc_scores, "pred_tgt": pred_tgt,
            "intervals": intervals, "pred_cal": pred_cal,
            "ys_cal": ys[cal_i], "n_cal": n_c}

m3_results = {}
for ts in STORES:
    r  = results[ts]
    m3 = method3_conformal(r["Xs"], r["ys"], r["Xt"], r["lo"], r["hi"])
    actual_cov = float((r["actual_errors"] <= m3["q"]).mean())
    m3["actual_coverage"] = actual_cov
    m3_results[ts] = m3

# ── Plot 1: Calibration nonconformity score distributions ─────────────────────
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5), sharey=False)
fig.suptitle("Method 3 - Calibration nonconformity score distributions", fontweight="bold")

for ax, ts, lbl, col in zip(axes, STORES, LABELS, COLORS):
    m3 = m3_results[ts]
    nc = m3["nc_scores"] / 1000
    ax.hist(nc, bins=20, color=col, alpha=0.7, edgecolor="white", density=True)
    ax.axvline(m3["q"]/1000, color="crimson", linewidth=2,
               label=f"q={m3['q']/1000:.1f}k")
    ax.axvline(results[ts]["actual_mae"]/1000, color="black", linewidth=1.5,
               linestyle="--", label=f"actual MAE={results[ts]['actual_mae']/1000:.1f}k")
    ax.set_title(f"{lbl}\ncoverage={m3['actual_coverage']:.0%}",
                 fontweight="bold", color=col, fontsize=9)
    ax.set_xlabel("Nonconformity score ($k)")
    ax.legend(fontsize=7)

plt.tight_layout(); plt.show()

# ── Plot 2: Coverage vs interval width ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Method 3 - Conformal intervals: coverage and width", fontweight="bold")

# Actual coverage vs target
ax = axes[0]
ax.axhline(0.90, color="black", linestyle="--", linewidth=1, alpha=0.5,
           label="Target coverage 90%")
ax.bar(range(len(STORES)), [m3_results[s]["actual_coverage"] for s in STORES],
       color=COLORS, alpha=0.8, edgecolor="white")
ax.set_xticks(range(len(STORES))); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Actual coverage"); ax.set_ylim(0, 1.1)
ax.set_title("Actual vs target (90%) coverage", fontweight="bold")
for i, ts in enumerate(STORES):
    cov = m3_results[ts]["actual_coverage"]
    ax.text(i, cov+0.02, f"{cov:.0%}", ha="center", fontsize=9, fontweight="bold")
ax.legend(fontsize=8)

# Interval width vs actual error distribution
ax = axes[1]
x = np.arange(len(STORES))
qs   = [m3_results[s]["q"]/1000       for s in STORES]
maes = [results[s]["actual_mae"]/1000  for s in STORES]
ax.bar(x-0.2, qs,   0.35, label="Conformal q (half-width)", color=COLORS, alpha=0.8)
ax.bar(x+0.2, maes, 0.35, label="Actual MAE", color=[c+"55" for c in COLORS],
       edgecolor=COLORS, linewidth=1.5)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("$ thousands"); ax.set_title("Interval half-width vs actual MAE", fontweight="bold")
ax.legend(fontsize=8)
for i, (q, mae) in enumerate(zip(qs, maes)):
    ratio = q / mae
    ax.text(i, max(q, mae)+0.5, f"q/MAE={ratio:.1f}x", ha="center", fontsize=7.5)

plt.tight_layout(); plt.show()

## 7. Visualising conformal intervals on actual predictions

For each store, plot predictions with ±q error bars against actual income.
Green points = actual value falls inside the interval.
Red points = actual value falls outside (coverage failures).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle("Conformal prediction intervals - actual vs predicted with coverage",
             fontweight="bold")

for ax, ts, lbl, col in zip(axes, STORES, LABELS, COLORS):
    m3 = m3_results[ts]
    y_true = results[ts]["yt"]
    preds  = m3["pred_tgt"]
    q      = m3["q"]
    lo_i   = preds - q
    hi_i   = preds + q
    covered = (y_true >= lo_i) & (y_true <= hi_i)

    # Sample 60 points for clarity
    idx_s = np.random.choice(len(y_true), min(60, len(y_true)), replace=False)
    idx_s = idx_s[np.argsort(y_true[idx_s])]

    ax.errorbar(range(len(idx_s)), preds[idx_s]/1000,
                yerr=q/1000, fmt="none", ecolor=col, alpha=0.3, elinewidth=0.8)
    ax.scatter(range(len(idx_s)), y_true[idx_s]/1000,
               c=["#27500A" if covered[i] else "#791F1F" for i in idx_s],
               s=18, zorder=3)
    ax.scatter(range(len(idx_s)), preds[idx_s]/1000, c=col,
               s=10, alpha=0.6, zorder=4, marker="^")

    cov_pct = covered.mean()
    ax.set_title(f"{lbl}\n{cov_pct:.0%} covered | q=+-${q/1000:.0f}k",
                 fontsize=9, fontweight="bold", color=col)
    ax.set_xlabel("Customer (sorted by income)")
    if ax == axes[0]: ax.set_ylabel("Income ($k)")

# Legend
green_patch = mpatches.Patch(color="#27500A", label="Actual (covered)")
red_patch   = mpatches.Patch(color="#791F1F", label="Actual (missed)")
fig.legend(handles=[green_patch, red_patch], loc="lower center",
           ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout(); plt.show()

print("\nWhy is Thrift coverage higher even though it has large shift?")
print("  Large shift -> wide calibration residuals -> large q -> wide intervals")
print("  Wider intervals are easier to cover -> higher coverage but less informative")

## 8. Method 4 — Density-ratio weighted residuals

**Core idea:** Not all source samples are equally representative of the target.
Reweight source residuals by how similar each source sample is to the target distribution.

**Importance weight formula:**
```
w(x) = p_target(x) / p_source(x)
```

Using Kernel Density Estimation:
```
log w(x) = log_kde_target(x) - log_kde_source(x)
w(x) = exp(log w(x) - max(log w))   # numerical stability
```

**Weighted MAE estimate:**
```
weighted_MAE = sum(w_i * |y_i - pred_i|) / sum(w_i)
```

Source samples that look like target customers get high weight,
so the error estimate reflects what we would observe on target-like data.

**Effective sample size** = 1 / sum((w_i / sum_w)^2)
Low effective n = most source samples look nothing like target -> unreliable estimate.

In [ ]:
def method4_density_ratio(Xs, ys, Xt, lo, hi, model, bw=1.5, seed=42):
    rng   = np.random.RandomState(seed)
    n_c   = int(0.2 * len(Xs))
    cal_i = rng.permutation(len(Xs))[-n_c:]
    Xs_c, ys_c = Xs[cal_i], ys[cal_i]

    kde_tgt = KernelDensity(bandwidth=bw, kernel="gaussian").fit(Xt)
    kde_src = KernelDensity(bandwidth=bw, kernel="gaussian").fit(Xs)

    log_w = kde_tgt.score_samples(Xs_c) - kde_src.score_samples(Xs_c)
    log_w -= log_w.max()
    w = np.exp(log_w); w = np.clip(w, 0.01, None); w /= w.mean()

    pred_c = lo + np.clip(model.predict(Xs_c),0,1)*(hi-lo)
    nc_c   = np.abs(ys_c - pred_c)
    eff_n  = 1.0 / np.sum((w/w.sum())**2)
    return {"est_mae": float(np.average(nc_c, weights=w)),
            "weights": w, "eff_n": eff_n, "nc_scores": nc_c}

m4_results = {}
for ts in STORES:
    r  = results[ts]
    m4 = method4_density_ratio(r["Xs"], r["ys"], r["Xt"], r["lo"], r["hi"], r["model"])
    m4_results[ts] = m4

# ── Plot: weight distributions per store ─────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle("Method 4 - Density-ratio weights: which source samples look like target?",
             fontweight="bold")

for i, (ts, lbl, col) in enumerate(zip(STORES, LABELS, COLORS)):
    m4 = m4_results[ts]
    # Row 1: weight distributions
    ax = axes[0, i]
    ax.hist(m4["weights"], bins=25, color=col, alpha=0.75, edgecolor="white",
            density=True)
    ax.axvline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.6,
               label="Equal weight")
    ax.set_title(f"{lbl}\nEff. n={m4['eff_n']:.0f}/200",
                 fontsize=9, fontweight="bold", color=col)
    ax.set_xlabel("Importance weight"); ax.legend(fontsize=7)
    if i == 0: ax.set_ylabel("Density")

    # Row 2: weighted vs unweighted residuals
    ax = axes[1, i]
    nc = m4["nc_scores"] / 1000
    w  = m4["weights"]
    ax.scatter(nc, w, c=col, s=15, alpha=0.5, edgecolors="none")
    ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.axvline(m4["est_mae"]/1000, color=col, linewidth=1.5, label=f"w.MAE=${m4['est_mae']/1000:.1f}k")
    ax.axvline(results[ts]["actual_mae"]/1000, color="black", linewidth=1.5,
               linestyle=":", label=f"actual=${results[ts]['actual_mae']/1000:.1f}k")
    ax.set_xlabel("Residual ($k)"); ax.set_title("Weight vs residual", fontsize=9)
    ax.legend(fontsize=7)
    if i == 0: ax.set_ylabel("Importance weight")

plt.tight_layout(); plt.show()

## 9. All four methods compared — which works best and when?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("All four methods: estimated vs actual MAE", fontweight="bold", fontsize=13)

# ── Panel 1: bar comparison ───────────────────────────────────────────────────
ax = axes[0, 0]
x  = np.arange(len(STORES)); w = 0.18
data_bars = [
    ("Actual",           [results[s]["actual_mae"]/1000     for s in STORES], "/", 1.0),
    ("M1 source CV",     [m1_results[s]["est_mae"]/1000     for s in STORES], "", 0.9),
    ("M2 shift-adj",     [m2_results[s]["est_mae"]/1000     for s in STORES], ".", 0.8),
    ("M4 density-ratio", [m4_results[s]["est_mae"]/1000     for s in STORES], "", 0.7),
]
offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]
for (lbl, vals, hatch, alpha), off in zip(data_bars, offsets):
    kw = dict(hatch=hatch) if hatch else {}
    ax.bar(x+off, vals, w, label=lbl, color=COLORS, alpha=alpha,
           edgecolor="white", **kw)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("MAE ($k)"); ax.set_title("MAE estimates vs actual", fontweight="bold")
ax.legend(fontsize=8)

# ── Panel 2: estimation error (|est - actual|) ─────────────────────────────────
ax = axes[0, 1]
err_m1 = [abs(results[s]["actual_mae"]-m1_results[s]["est_mae"])/1000 for s in STORES]
err_m2 = [abs(results[s]["actual_mae"]-m2_results[s]["est_mae"])/1000 for s in STORES]
err_m4 = [abs(results[s]["actual_mae"]-m4_results[s]["est_mae"])/1000 for s in STORES]
offsets3 = [-w, 0, w]
for errs, lbl, off, hatch in [(err_m1,"M1",offsets3[0],""),
                               (err_m2,"M2",offsets3[1],"."),
                               (err_m4,"M4",offsets3[2],"")]:
    kw = dict(hatch=hatch) if hatch else {}
    ax.bar(x+off, errs, w*1.8, label=lbl, color=COLORS, alpha=0.8,
           edgecolor="white", **kw)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("|Estimated - Actual| MAE ($k)")
ax.set_title("Estimation error per method", fontweight="bold"); ax.legend(fontsize=8)

# ── Panel 3: conformal coverage ───────────────────────────────────────────────
ax = axes[1, 0]
coverages = [m3_results[s]["actual_coverage"] for s in STORES]
widths    = [m3_results[s]["q"]/1000          for s in STORES]
ax.bar(x, coverages, 0.4, color=COLORS, alpha=0.8, edgecolor="white", label="Coverage")
ax.axhline(0.90, color="crimson", linestyle="--", linewidth=1.5, label="Target 90%")
ax2 = ax.twinx()
ax2.plot(x, widths, "o--", color="black", linewidth=1.5, markersize=6, label="Half-width ($k)")
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Coverage"); ax2.set_ylabel("Interval half-width ($k)")
ax.set_title("M3 Conformal: coverage vs interval width", fontweight="bold")
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labs1+labs2, fontsize=8)

# ── Panel 4: KS shift vs method reliability ───────────────────────────────────
ax = axes[1, 1]
ks_vals  = [results[s]["avg_ks"]                for s in STORES]
m1_errs  = [abs(results[s]["actual_mae"]-m1_results[s]["est_mae"])/results[s]["actual_mae"]*100
            for s in STORES]
m4_errs  = [abs(results[s]["actual_mae"]-m4_results[s]["est_mae"])/results[s]["actual_mae"]*100
            for s in STORES]
ax.scatter(ks_vals, m1_errs, c=COLORS, s=120, marker="o", zorder=5, label="M1 error %")
ax.scatter(ks_vals, m4_errs, c=COLORS, s=120, marker="^", zorder=5, label="M4 error %")
for i, (ks, e1, e4, lbl) in enumerate(zip(ks_vals, m1_errs, m4_errs, LABELS)):
    ax.annotate(lbl, (ks, max(e1,e4)), textcoords="offset points",
                xytext=(5, 3), fontsize=8, color=COLORS[i])

ks_line = np.linspace(0.1, 0.7, 50)
ax.axvline(0.3, color="darkorange", linestyle="--", alpha=0.6, linewidth=1)
ax.axvline(0.5, color="crimson",    linestyle="--", alpha=0.6, linewidth=1)
ax.set_xlabel("Avg KS shift"); ax.set_ylabel("Estimation error (%)")
ax.set_title("KS shift vs method reliability", fontweight="bold")
ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

## 10. Decision guide — which method to use?

Based on the avg KS statistic (measure feature shift first):

In [ ]:
# Summary table
rows = []
for ts, lbl in zip(STORES, LABELS):
    act   = results[ts]["actual_mae"]
    m1e   = m1_results[ts]["est_mae"]
    m2e   = m2_results[ts]["est_mae"]
    m3q   = m3_results[ts]["q"]
    m3cov = m3_results[ts]["actual_coverage"]
    m4e   = m4_results[ts]["est_mae"]
    ks    = results[ts]["avg_ks"]
    eff_n = m4_results[ts]["eff_n"]

    if ks < 0.25:  rec = "M1 or M4 — source CV reliable"
    elif ks < 0.45: rec = "M4 for point est., M3 for interval"
    else:           rec = "M3 only — report +-interval, not point MAE"

    rows.append({
        "Store": lbl,
        "Actual MAE": f"${act:,.0f}",
        "M1 est.": f"${m1e:,.0f} (err=${abs(act-m1e):,.0f})",
        "M2 est.": f"${m2e:,.0f} (err=${abs(act-m2e):,.0f})",
        "M3 interval": f"+-${m3q:,.0f} ({m3cov:.0%} cov.)",
        "M4 est.": f"${m4e:,.0f} (err=${abs(act-m4e):,.0f})",
        "Avg KS": f"{ks:.3f}",
        "Eff. n (M4)": f"{eff_n:.0f}/200",
        "Recommendation": rec,
    })

display(pd.DataFrame(rows).set_index("Store"))

# ── Decision flow chart as annotated plot ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlim(0, 10); ax.set_ylim(0, 4); ax.axis("off")
ax.set_title("Decision guide: which uncertainty method to use?",
             fontweight="bold", fontsize=12)

boxes = [
    (1,   2, "Measure\navg KS shift",  "#185FA5"),
    (3.5, 3, "KS < 0.25\n(small shift)","#1D9E75"),
    (3.5, 2, "KS 0.25-0.45\n(medium)","#854F0B"),
    (3.5, 1, "KS > 0.45\n(large shift)","#A32D2D"),
    (7,   3, "Use M1 (source CV)\nor M4 (density-ratio)","#1D9E75"),
    (7,   2, "Use M4 for point est.\nM3 for +-interval","#854F0B"),
    (7,   1, "Use M3 only\nReport +-interval","#A32D2D"),
]
for bx, by, btxt, bcol in boxes:
    ax.text(bx, by, btxt, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.5", facecolor=bcol+"33",
                      edgecolor=bcol, linewidth=2))

arrows = [(1.6, 2, 2.7, 3), (1.6, 2, 2.7, 2), (1.6, 2, 2.7, 1),
          (4.5, 3, 5.7, 3), (4.5, 2, 5.7, 2), (4.5, 1, 5.7, 1)]
for x1,y1,x2,y2 in arrows:
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))

plt.tight_layout(); plt.show()

print("\nKey takeaways:")
print("  1. For KS < 0.25: source CV (M1) is reliable. No adjustment needed.")
print("  2. For KS 0.25-0.45: use density-ratio (M4) for best point estimate.")
print("  3. For KS > 0.45: only conformal prediction (M3) is reliable.")
print("     Report a +-interval, NOT a single point MAE estimate.")
print("  4. The conformal interval is always available regardless of KS.")
print("     When in doubt, always pair any point estimate with M3's interval.")

## 11. Per-customer uncertainty — where is the model most and least confident?

The conformal interval is the same width for all customers (it uses a global quantile q).
But we can estimate per-customer uncertainty by looking at how close each customer is to
the training distribution — customers far from training data should have higher uncertainty.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle("Per-customer uncertainty: distance to source training data",
             fontweight="bold")

for ax, ts, lbl, col in zip(axes, STORES, LABELS, COLORS):
    r   = results[ts]
    Xt  = r["Xt"]; yt = r["yt"]; preds = r["preds"]
    errs = r["actual_errors"]

    # Per-customer distance to nearest source centroid (proxy for uncertainty)
    Xp_src = pca_ref.transform(r["Xs"])
    Xp_tgt = pca_ref.transform(Xt)
    src_cen = Xp_src.mean(axis=0)
    dist_to_src = np.linalg.norm(Xp_tgt - src_cen, axis=1)

    # Does distance predict error?
    sc_plot = ax.scatter(dist_to_src, errs/1000, c=yt/1000, cmap="RdYlGn",
                          s=15, alpha=0.6, edgecolors="none",
                          vmin=yt.min()/1000, vmax=yt.max()/1000)
    # Trend line
    z = np.polyfit(dist_to_src, errs/1000, 1)
    xs = np.linspace(dist_to_src.min(), dist_to_src.max(), 50)
    ax.plot(xs, np.poly1d(z)(xs), color=col, linewidth=2, alpha=0.8)
    sp, _ = spearmanr(dist_to_src, errs)
    ax.set_title(f"{lbl}\nr={sp:.2f}", fontsize=9, fontweight="bold", color=col)
    ax.set_xlabel("Distance to source"); ax.set_ylabel("Error ($k)")

plt.colorbar(sc_plot, ax=axes[-1], label="Income ($k)", shrink=0.8)
plt.tight_layout(); plt.show()

print("Interpretation:")
print("  Positive r = customers farther from source training data have higher errors.")
print("  This validates that feature distance is a proxy for prediction uncertainty.")
print("  Customers in the 'overlap zone' (low distance) have reliably lower errors.")